In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
data_df = pd.read_csv('datasets/training_set_VU_DM.csv')

In [6]:
data_df

,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool
0,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,893,3,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
1,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,10404,4,4.0,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
2,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,21315,3,4.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
3,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,27348,2,4.0,...,NaN,NaN,NaN,NaN,-1.0,0.0,5.0,0,NaN,0
4,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,29604,4,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4958342,332785,2013-06-30 19:55:18,5,219,NaN,NaN,219,77700,3,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0
4958343,332785,2013-06-30 19:55:18,5,219,NaN,NaN,219,88083,3,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0
4958344,332785,2013-06-30 19:55:18,5,219,NaN,NaN,219,94508,3,3.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0
4958345,332785,2013-06-30 19:55:18,5,219,NaN,NaN,219,128360,3,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,157.84,1


In [ ]:
# OUTLIER

# MISSING VALUES, prop: per prop_id
# visitor_hist_starrating: super sparse
# visitor_hist_adr_usd: super sparse
# prop_review_score: handled by feature engineering
# prop_location_score2
# srch_query_affinity_score: super sparse
# orig_destination_distance: 32 % sparse
# comp_rate: super sparse, do sth depending on sparsity
# comp_inv: super sparse

# position


# FEATURE ENGINEERING
# date_time -> weekday, season, etc
# prop_star_rating -> prop_start_rating_hiddne (bool), prop_star_rating
# prop_review_score -> prop_no_review (bool), prop_review_unavailable, prop_review_score
# prop_log_historical_price -> remove log
# price_usd: log
# srch_query_affinity_score -> handle 0s, remove log
# orig_destination_distance -> orig_not_cancelled (bool), orig_destination_distance



In [8]:
def _add_column_indicating_missing_values(df, column_name):
    df[column_name+ 'missing'] = data_df[column_name].isnull().astype(int)
    return df

# prop_review_score
def add_column_indicatig_zero_values(df, column_name):
    df['zero' + column_name] = (data_df[column_name] == 0).astype(int)
    return df

# visitor_hist_starrating (fill with prop_starrating)
# visitor_hist_adr_usd (fill with price_usd (later use improved version)
def fill_missing_from_other_column(df, column_name, fill_from):
    df = _add_column_indicating_missing_values(df, column_name)
    df[column_name] = df[column_name].fillna(df[fill_from])
    return df

# prop_review_score, prop_location_score2, srch_query_affinity_score, orig_destination_distance
def fill_missing_with_mean(df, column_name):
    df = _add_column_indicating_missing_values(df, column_name)
    df[column_name] = df[column_name].fillna(df[column_name].mean())
    return df

# comp_rate, comp_inv, comp_rate_percent_diff
def fill_missing_with_median(df, column_name):
    df = _add_column_indicating_missing_values(df, column_name)
    df[column_name] = df[column_name].fillna(df[column_name].median())
    return df

In [9]:
# copy data
data_df_no_missing = data_df.copy()

# Impute missing values
# Zero values
data_df_no_missing = add_column_indicatig_zero_values(data_df, 'prop_review_score')

# Fill from other column
data_df_no_missing = fill_missing_from_other_column(data_df, 'visitor_hist_starrating', 'prop_starrating')
data_df_no_missing = fill_missing_from_other_column(data_df, 'visitor_hist_adr_usd', 'price_usd')


# Fill with mean
data_df_no_missing = fill_missing_with_mean(data_df, 'prop_review_score')
data_df_no_missing = fill_missing_with_mean(data_df, 'prop_location_score2')
data_df_no_missing = fill_missing_with_mean(data_df, 'srch_query_affinity_score')
data_df_no_missing = fill_missing_with_mean(data_df, 'orig_destination_distance')

for i in range(1, 9):
    data_df_no_missing = fill_missing_with_median(data_df, 'comp' + str(i) + '_rate')
    data_df_no_missing = fill_missing_with_median(data_df, 'comp' + str(i) + '_inv')
    data_df_no_missing = fill_missing_with_median(data_df, 'comp' + str(i) + '_rate_percent_diff')

In [10]:
# Save
data_df_no_missing.to_csv('datasets/training_set_no_missing.csv', index=False)

In [ ]:
# check if everyting in between bounds, otherwise prune (and add column indicating pruned)
data_df_no_missing